# OmniGeoFusion — Stage 2: Fine-tuning

Task-specific fine-tuning on top of SSL pre-trained backbone.

**Prerequisites:** Run `01_ssl_pretrain.ipynb` first  
**Runtime:** T4 GPU  
**Duration:** ~2-3 hours per task (15 epochs total)  
**Tasks:** urban_change → flood_damage → agriculture

### Training Strategy
- **Phase 1** (5 epochs): Frozen backbone, train head only
- **Phase 2** (10 epochs): Unfreeze last 4 optical layers

In [ ]:
# ── Cell 1: Mount Drive + Check GPU ──
from google.colab import drive
drive.mount('/gdrive')

import torch, os
print(f'GPU:  {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB')

# Check SSL checkpoint exists
SSL_CKPT = '/gdrive/MyDrive/omnigeofusion/checkpoints/ssl/ssl_best.pt'
if os.path.exists(SSL_CKPT):
    ckpt = torch.load(SSL_CKPT, map_location='cpu')
    print(f'✅ SSL checkpoint: epoch {ckpt.get("epoch")} loss {ckpt.get("best_loss", "N/A")}')
else:
    print('⚠️  No SSL checkpoint — run 01_ssl_pretrain.ipynb first')

os.makedirs('/gdrive/MyDrive/omnigeofusion/checkpoints/finetune', exist_ok=True)
os.makedirs('/gdrive/MyDrive/omnigeofusion/checkpoints/best', exist_ok=True)

In [ ]:
# ── Cell 2: Setup Virtual Environment ──
import subprocess, os

VENV_PYTHON = '/content/venv/bin/python3'
VENV_PIP    = '/content/venv/bin/pip'

# Skip if venv already exists from SSL notebook
if os.path.exists(VENV_PYTHON):
    print('✅ Venv already exists from SSL session')
else:
    os.system('apt-get install -y libgeos-dev libgdal-dev -q')
    os.environ['MPLBACKEND'] = 'agg'
    os.system('python3 -m venv /content/venv')
    os.system('curl -sS https://bootstrap.pypa.io/get-pip.py | /content/venv/bin/python3')

    def install(packages):
        r = subprocess.run(
            [VENV_PIP, 'install', '-q'] + packages,
            capture_output=True, text=True
        )
        if r.returncode != 0:
            print(f'Error: {r.stderr[-500:]}')
        return r.returncode == 0

    print('Step 1: terratorch...')
    install(['terratorch'])
    print('Step 2: torch...')
    install(['--force-reinstall', 'torch==2.2.0', 'torchvision==0.17.0'])
    print('Step 3: dependencies...')
    install([
        'transformers==4.40.0', 'timm', 'einops',
        'mlflow==2.14.3', 'boto3', 'rasterio',
        'scipy', 'pyyaml', 'huggingface_hub==0.20.3',
        'pandas', 'tqdm', 'pyproj', 'scikit-learn',
    ])
    print('Step 4: pin versions...')
    install(['numpy==1.26.4', 'protobuf==3.20.3', 'setuptools==69.5.1'])
    os.system('pip install -q boto3 rasterio')

print('✅ Venv ready')

In [ ]:
# ── Cell 3: Setup Repo ──
import os

REPO_DIR   = '/content/omnigeofusion'
REPO_DRIVE = '/gdrive/MyDrive/omnigeofusion/repo'
BUCKET     = 'omnigeofusion-data-288528696055'

if os.path.exists(f'{REPO_DIR}/src'):
    print('✅ Repo already in /content')
elif os.path.exists(f'{REPO_DRIVE}/src'):
    os.system(f'cp -r {REPO_DRIVE} {REPO_DIR}')
    print('✅ Repo copied from Drive')
else:
    os.system(f'aws s3 cp s3://{BUCKET}/repo/omnigeofusion.tar.gz /tmp/omnigeofusion.tar.gz')
    os.makedirs(REPO_DIR, exist_ok=True)
    os.system('tar -xzf /tmp/omnigeofusion.tar.gz -C /content/')
    print('✅ Repo downloaded from S3')

print(f'Contents: {os.listdir(REPO_DIR)[:6]}')

In [ ]:
# ── Cell 4: AWS Credentials + MLflow ──
import os, subprocess
from google.colab import userdata

VENV_PYTHON = '/content/venv/bin/python3'
AWS_KEY     = userdata.get('AWS_ACCESS_KEY_ID')
AWS_SECRET  = userdata.get('AWS_SECRET_ACCESS_KEY')
MLFLOW_URI  = 'http://54.83.1.56:5000'

os.environ['AWS_ACCESS_KEY_ID']     = AWS_KEY
os.environ['AWS_SECRET_ACCESS_KEY'] = AWS_SECRET
os.environ['AWS_DEFAULT_REGION']    = 'us-east-1'
os.environ['MLFLOW_TRACKING_URI']   = MLFLOW_URI

# Test S3
import boto3
s3 = boto3.client('s3', region_name='us-east-1')
resp = s3.list_objects_v2(
    Bucket='omnigeofusion-data-288528696055',
    Prefix='netherlands/targets/', MaxKeys=3
)
print(f'✅ S3: {resp["KeyCount"]} target files found')
print(f'✅ MLflow: {MLFLOW_URI}')

In [ ]:
# ── Cell 5: Build Task-specific Fine-tuning Index ──
# Downloads target files + matches with patch paths
import boto3, json, os, random
from pathlib import Path
from google.colab import userdata

AWS_KEY    = userdata.get('AWS_ACCESS_KEY_ID')
AWS_SECRET = userdata.get('AWS_SECRET_ACCESS_KEY')

s3 = boto3.client('s3',
    region_name='us-east-1',
    aws_access_key_id=AWS_KEY,
    aws_secret_access_key=AWS_SECRET
)
BUCKET   = 'omnigeofusion-data-288528696055'
DATA_DIR = Path('/content/omnigeofusion_data')

TASK_CONFIG = {
    'urban_change': {'area': 'amsterdam', 'tile': '31UFT'},
    'flood_damage': {'area': 'rotterdam',  'tile': '31UFS'},
    'agriculture':  {'area': 'flevoland',  'tile': '31UGU'},
}

MAX_PATCHES = 400
TRAIN_RATIO = 0.85

def get_index(modality, area):
    key  = f'netherlands/{modality}/index/{area}_index.json'
    path = DATA_DIR / modality / 'index' / f'{area}_index.json'
    path.parent.mkdir(parents=True, exist_ok=True)
    if not path.exists():
        s3.download_file(BUCKET, key, str(path))
    return json.loads(path.read_text())

def get_targets(task, area):
    key  = f'netherlands/targets/index/{area}_targets.json'
    path = DATA_DIR / 'targets' / f'{area}_targets.json'
    path.parent.mkdir(parents=True, exist_ok=True)
    if not path.exists():
        s3.download_file(BUCKET, key, str(path))
    return json.loads(path.read_text())

def download_patches(modality, area, index, max_n):
    downloaded = []
    for entry in index[:max_n]:
        local = DATA_DIR / entry['s3_key']
        if not local.exists():
            local.parent.mkdir(parents=True, exist_ok=True)
            try:
                s3.download_file(BUCKET, entry['s3_key'], str(local))
            except:
                continue
        entry['local_path'] = str(local)
        downloaded.append(entry)
    return downloaded

print('Building fine-tuning indexes per task...')

for task, cfg in TASK_CONFIG.items():
    area = cfg['area']
    tile = cfg['tile']
    print(f'\nTask: {task} ({area})')

    # Load indexes
    s2_idx   = get_index('sentinel2', area)
    sar_idx  = get_index('sentinel1', area)
    lid_idx  = get_index('lidar',     area)
    thm_idx  = get_index('thermal',   area)
    targets  = get_targets(task, area)

    # Download patches
    print(f'  Downloading S2...')
    s2_dl  = download_patches('sentinel2', area, s2_idx,  MAX_PATCHES)
    print(f'  Downloading SAR...')
    sar_dl = download_patches('sentinel1', area, sar_idx, MAX_PATCHES)
    print(f'  Downloading LiDAR...')
    lid_dl = download_patches('lidar',     area, lid_idx, MAX_PATCHES)
    print(f'  Downloading Thermal...')
    thm_dl = download_patches('thermal',   area, thm_idx, MAX_PATCHES)

    # Build lookup by (row, col)
    lookup = {}
    for e in s2_dl:
        k = (e['row'], e['col'])
        lookup[k] = {
            's2_t1_path':   e['local_path'],
            'patch_id':     e['patch_id'],
            'tile':         tile,
            'area':         area,
            'date':         e['date'],
            'row':          e['row'],
            'col':          e['col'],
            'day_gap':      90,
        }

    for mod, dl, path_key in [
        ('sentinel1', sar_dl, 'sar_path'),
        ('lidar',     lid_dl, 'lidar_path'),
        ('thermal',   thm_dl, 'thermal_path'),
    ]:
        for e in dl:
            k = (e['row'], e['col'])
            if k in lookup:
                lookup[k][path_key] = e['local_path']

    # Add targets
    tgt_lookup = {t['patch_id']: t for t in targets}
    samples = []
    for k, sample in lookup.items():
        pid = sample['patch_id']
        if pid in tgt_lookup:
            sample.update(tgt_lookup[pid])
            samples.append(sample)

    random.shuffle(samples)
    split      = int(len(samples) * TRAIN_RATIO)
    train_idx  = samples[:split]
    val_idx    = samples[split:]

    task_dir = DATA_DIR / task
    task_dir.mkdir(parents=True, exist_ok=True)
    (task_dir / 'train_index.json').write_text(json.dumps(train_idx, indent=2))
    (task_dir / 'val_index.json').write_text(json.dumps(val_idx, indent=2))

    print(f'  ✅ {task}: train={len(train_idx)} val={len(val_idx)}')

print('\n✅ All task indexes built')

In [ ]:
# ── Cell 6: Update Training Config ──
import yaml, os

os.chdir('/content/omnigeofusion')
train_cfg = yaml.safe_load(open('configs/training_config.yaml'))

train_cfg['checkpoints'] = {
    'ssl_dir':      '/gdrive/MyDrive/omnigeofusion/checkpoints/ssl',
    'finetune_dir': '/gdrive/MyDrive/omnigeofusion/checkpoints/finetune',
    'best_dir':     '/gdrive/MyDrive/omnigeofusion/checkpoints/best',
    'gdrive_path':  '/gdrive/MyDrive/omnigeofusion/checkpoints',
}
train_cfg['finetune'] = {
    **train_cfg.get('finetune', {}),
    'phase1_epochs': 5,
    'phase2_epochs': 10,
    'batch_size':    8,
}
train_cfg['mlflow'] = {
    'tracking_uri':    'http://54.83.1.56:5000',
    'experiment_name': 'omnigeofusion-finetune',
}

with open('configs/training_config.yaml', 'w') as f:
    yaml.dump(train_cfg, f, default_flow_style=False)

print('✅ Config updated')
print(f'   Phase 1: {train_cfg["finetune"]["phase1_epochs"]} epochs (frozen)')
print(f'   Phase 2: {train_cfg["finetune"]["phase2_epochs"]} epochs (partial unfreeze)')

In [ ]:
# ── Cell 7: Fine-tune All 3 Tasks ──
import subprocess, os
from google.colab import userdata

VENV_PYTHON = '/content/venv/bin/python3'
AWS_KEY     = userdata.get('AWS_ACCESS_KEY_ID')
AWS_SECRET  = userdata.get('AWS_SECRET_ACCESS_KEY')
SSL_CKPT    = '/gdrive/MyDrive/omnigeofusion/checkpoints/ssl/ssl_best.pt'
CKPT_DIR    = '/gdrive/MyDrive/omnigeofusion/checkpoints/finetune'

env = os.environ.copy()
env['MPLBACKEND']            = 'agg'
env['AWS_ACCESS_KEY_ID']     = AWS_KEY
env['AWS_SECRET_ACCESS_KEY'] = AWS_SECRET
env['AWS_DEFAULT_REGION']    = 'us-east-1'
env['MLFLOW_TRACKING_URI']   = 'http://54.83.1.56:5000'
env['PYTHONPATH']            = '/content/omnigeofusion'

os.chdir('/content/omnigeofusion')

for task in ['urban_change', 'flood_damage', 'agriculture']:
    print(f'\n{"="*50}')
    print(f'Fine-tuning: {task}')
    print(f'{"="*50}')

    # Check for existing checkpoint to resume
    latest = f'{CKPT_DIR}/{task}/finetune_latest.pt'

    cmd = [
        VENV_PYTHON, '-m', 'src.training.finetune',
        '--task',           task,
        '--model-config',   'configs/model_config.yaml',
        '--train-config',   'configs/training_config.yaml',
        '--data-dir',       '/content/omnigeofusion_data',
        '--max-samples',    '400',
    ]

    if os.path.exists(latest):
        print(f'⚠️  Resuming {task} from {latest}')
        cmd += ['--resume', latest]
    elif os.path.exists(SSL_CKPT):
        print(f'Loading SSL backbone from {SSL_CKPT}')
        cmd += ['--ssl-checkpoint', SSL_CKPT]

    process = subprocess.Popen(
        cmd, env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True, bufsize=1
    )
    for line in process.stdout:
        print(line, end='', flush=True)
    process.wait()
    print(f'\n{task} exited: {process.returncode}')

print('\n✅ All tasks fine-tuned')

In [ ]:
# ── Cell 8: Verify Checkpoints + Upload to S3 ──
import os, subprocess, boto3
from google.colab import userdata

AWS_KEY    = userdata.get('AWS_ACCESS_KEY_ID')
AWS_SECRET = userdata.get('AWS_SECRET_ACCESS_KEY')

s3 = boto3.client('s3',
    region_name='us-east-1',
    aws_access_key_id=AWS_KEY,
    aws_secret_access_key=AWS_SECRET
)
BUCKET    = 'omnigeofusion-data-288528696055'
BEST_DIR  = '/gdrive/MyDrive/omnigeofusion/checkpoints/best'
CKPT_DIR  = '/gdrive/MyDrive/omnigeofusion/checkpoints/finetune'

print('=== Fine-tuning Checkpoints ===')
for task in ['urban_change', 'flood_damage', 'agriculture']:
    task_dir = f'{CKPT_DIR}/{task}'
    if os.path.exists(task_dir):
        files = os.listdir(task_dir)
        print(f'\n{task}:')
        for f in sorted(files):
            size = os.path.getsize(f'{task_dir}/{f}') / 1e6
            print(f'  {f}: {size:.1f}MB')

        # Upload best checkpoint to S3
        best = f'{BEST_DIR}/{task}_best.pt'
        if os.path.exists(best):
            s3_key = f'models/finetune/{task}_best.pt'
            s3.upload_file(best, BUCKET, s3_key)
            print(f'  ✅ Uploaded: s3://{BUCKET}/{s3_key}')
    else:
        print(f'\n{task}: No checkpoints yet')

print('\n→ Ready for inference: 03_inference.ipynb')